# BTC and ETH in the premium portfolio

Compare the core and supplemental books on BTC with the same books on BTC and ETH in January 2025. Both runs use the portfolio rules. The two-chain run starts with 1 BTC and 20 ETH. Each combo is sized from total equity in its settlement coin. Correlated underlyings share the larger Equation 8.1 fraction.

Results use BTC as the reporting currency. The starting capital differs between runs. Use returns and risk ratios for the comparison. Changes in the ETH/BTC rate also change the value of the ETH capital.

Set `SBT2_SERVER_TOKEN` and, if needed, `SBT2_SERVER_URL`. The server needs both option chains and a year of index and DVOL history.

In [ ]:
import os

import pandas as pd

from sbt2.lab import Lab
from sbt2.strategies.options.premium_portfolio import PremiumPortfolio

url = os.environ.get("SBT2_SERVER_URL", "ws://127.0.0.1:8765")
lab = await Lab.connect(url, os.environ["SBT2_SERVER_TOKEN"])

In [ ]:
spec = {
    "instruments": ["BTC-OPTIONS.DERIBIT"],
    "period": ["2025-01-01", "2025-02-05"],
    "execution": {
        "fee_model": {
            "kind": "capped_option",
            "config": {
                "maker_rate": "0.0003",
                "taker_rate": "0.0003",
                "cap_rate": "0.125",
            },
        }
    },
    "capital": "1 BTC",
    "split": {"validation_start": "2025-02-01", "test_start": "2025-02-03"},
    "part": "train",
    "chain": {"min_dte": 0, "max_dte": 30, "strike_band": "0.3"},
    "params": {"core_dte": 14, "supplemental_dte": 14, "close_dte": 3},
    "risk": {"short_premium": {}},
}

btc = await lab.run(spec, PremiumPortfolio)
both = await lab.run(
    {
        **spec,
        "instruments": ["BTC-OPTIONS.DERIBIT", "ETH-OPTIONS.DERIBIT"],
        "capital": ["1 BTC", "20 ETH"],
    },
    PremiumPortfolio,
)
pd.concat([btc.table(), both.table()], keys=["BTC", "BTC + ETH"])

Compare the trade statistics and the option tearsheets of both runs.

In [ ]:
pd.DataFrame(
    {
        "BTC": await btc[0].trade_statistics(),
        "BTC + ETH": await both[0].trade_statistics(),
    }
)

In [ ]:
await btc[0].tearsheet()

In [ ]:
await both[0].tearsheet()

In [ ]:
await lab.close()